# Stage 3 · Monthly Panel and Event-Study DiD

Reconstructs monthly rankings to identify and exclude late top-N entrants,
collects the monthly release panel (crash-safe, resumable), and estimates
the pooled difference-in-differences and the dynamic event study with
power reporting (paper Secs. 3.6 and 4.1).

---
## Cell 0 — Crash-safe infrastructure (run first, every session)

In [ ]:
# ============ CELL 0 — CRASH-SAFE INFRASTRUCTURE ============
"""
Crash-safe collection primitives for the MSR 2027 pipeline.

Design rules, each learned from a way Colab actually fails:
  1. APPEND-ONLY JSONL, never rewrite-in-place. A crash mid-write loses at
     most the last partial line, not the whole file.
  2. ATOMIC writes for any file that must be rewritten (tmp + os.replace).
  3. FAILURES ARE RECORDED, so permanently-dead items are not retried forever.
  4. MIRRORED to Drive but WRITTEN to local disk, so a dropped Drive mount
     mid-run cannot corrupt the checkpoint.
  5. RESUME IS O(n), not O(n^2).
"""
from __future__ import annotations
import json, os, shutil, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor


def atomic_write_bytes(path: Path, data: bytes) -> None:
    tmp = path.with_suffix(path.suffix + '.tmp')
    with open(tmp, 'wb') as f:
        f.write(data); f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)


class ResumableCollector:
    """Append-only, crash-safe, resumable map over a list of string keys."""

    def __init__(self, name, local_dir='/content/msr_state', mirror_dir=None,
                 workers=12, flush_every=25, mirror_every=100, mirror_seconds=60, max_attempts=2):
        self.name = name
        self.local = Path(local_dir); self.local.mkdir(parents=True, exist_ok=True)
        self.mirror = Path(mirror_dir) if mirror_dir else None
        if self.mirror: self.mirror.mkdir(parents=True, exist_ok=True)
        self.workers, self.flush_every = workers, flush_every
        self.mirror_every, self.max_attempts = mirror_every, max_attempts
        self.mirror_seconds = mirror_seconds
        self._last_mirror = time.time()
        self.data_path = self.local/f'{name}.jsonl'
        self.fail_path = self.local/f'{name}.failed.jsonl'
        self._restore_from_mirror()
        self.done, self.failed = self._load()

    # ---- mirror handling -------------------------------------------------
    def _restore_from_mirror(self):
        """If local disk was wiped (new runtime) but Drive has state, pull it back."""
        if not self.mirror: return
        for p in (self.data_path, self.fail_path):
            m = self.mirror/p.name
            if m.exists() and (not p.exists() or m.stat().st_size > p.stat().st_size):
                shutil.copy2(m, p)

    def _push_mirror(self):
        if not self.mirror: return
        try:
            for p in (self.data_path, self.fail_path):
                if p.exists(): shutil.copy2(p, self.mirror/p.name)
        except Exception as e:
            print(f'  [mirror warning: {e}]')   # never fatal

    # ---- state -----------------------------------------------------------
    def _load(self):
        done, failed = {}, {}
        if self.data_path.exists():
            with open(self.data_path) as f:
                for line in f:
                    try:
                        r = json.loads(line)
                        done[r['_key']] = r
                    except Exception:
                        continue          # truncated final line after a crash
        if self.fail_path.exists():
            with open(self.fail_path) as f:
                for line in f:
                    try:
                        r = json.loads(line); failed[r['_key']] = r.get('n', 1)
                    except Exception:
                        continue
        return done, failed

    def pending(self, keys):
        """O(n) resume: skip completed and permanently-failed keys."""
        d, fl, m = self.done, self.failed, self.max_attempts
        return [k for k in keys if k not in d and fl.get(k, 0) < m]

    # ---- the run loop ----------------------------------------------------
    def run(self, keys, fn, progress=True):
        """Workers write their OWN results the moment they complete.

        The earlier version wrote from the consumer side of ThreadPoolExecutor.map().
        That is wrong: map() submits everything eagerly, so workers can finish
        thousands of items while the consumer lags behind, and every one of those
        is lost on a hard kill. Writing inside the worker, under a lock, means a
        completed item is durable the moment it exists.
        """
        import threading
        todo = self.pending(keys)
        print(f'[{self.name}] total={len(keys):,}  done={len(self.done):,}  '
              f'failed={len(self.failed):,}  todo={len(todo):,}')
        if not todo:
            print(f'[{self.name}] nothing to do — already complete.')
            return self.frame()

        lock = threading.Lock()
        fd = open(self.data_path, 'a'); ff = open(self.fail_path, 'a')
        counters = {'ok': 0, 'bad': 0, 'since_flush': 0, 'since_mirror': 0}
        t0 = time.time()

        def worker(key):
            try:
                rec = fn(key)
            except Exception:
                rec = None
            with lock:
                if rec is None:
                    counters['bad'] += 1
                    n = self.failed.get(key, 0) + 1
                    ff.write(json.dumps({'_key': key, 'n': n}) + '\n')
                    self.failed[key] = n
                else:
                    counters['ok'] += 1
                    r = dict(rec); r['_key'] = key
                    fd.write(json.dumps(r, default=str) + '\n')
                    self.done[key] = r
                counters['since_flush'] += 1
                counters['since_mirror'] += 1
                if counters['since_flush'] >= self.flush_every:
                    fd.flush(); os.fsync(fd.fileno())
                    ff.flush(); os.fsync(ff.fileno())
                    counters['since_flush'] = 0
                due = (counters['since_mirror'] >= self.mirror_every or
                       time.time() - self._last_mirror >= self.mirror_seconds)
                if due:
                    fd.flush(); ff.flush(); self._push_mirror()
                    counters['since_mirror'] = 0; self._last_mirror = time.time()
            return key

        try:
            from tqdm.auto import tqdm
            wrap = (lambda it: tqdm(it, total=len(todo))) if progress else (lambda it: it)
        except Exception:
            wrap = lambda it: it

        try:
            with ThreadPoolExecutor(max_workers=self.workers) as ex:
                for _ in wrap(ex.map(worker, todo)):
                    pass
        except KeyboardInterrupt:
            print(f'\n[{self.name}] INTERRUPTED — progress is safe. Re-run to resume.')
        finally:
            try:
                fd.flush(); os.fsync(fd.fileno()); fd.close()
                ff.flush(); os.fsync(ff.fileno()); ff.close()
            except Exception: pass
            self._push_mirror()

        print(f'[{self.name}] +{counters["ok"]:,} ok, +{counters["bad"]:,} failed '
              f'in {time.time()-t0:.0f}s -> {len(self.done):,} total')
        return self.frame()

    def frame(self):
        import pandas as pd
        if not self.done: return pd.DataFrame()
        df = pd.DataFrame(list(self.done.values()))
        return df.rename(columns={'_key': 'project'})


class CellState:
    """Records which cells completed, so a fresh runtime tells you where you were."""
    def __init__(self, path):
        self.path = Path(path); self.state = {}
        if self.path.exists():
            try: self.state = json.loads(self.path.read_text())
            except Exception: self.state = {}

    def mark(self, cell):
        self.state[str(cell)] = time.strftime('%Y-%m-%d %H:%M:%SZ', time.gmtime())
        atomic_write_bytes(self.path, json.dumps(self.state, indent=2).encode())

    def report(self):
        if not self.state:
            print('No previous run recorded — start at cell 1.'); return
        print('Previously completed cells:')
        for k in sorted(self.state, key=lambda x: int(x)):
            print(f'   cell {k}: {self.state[k]}')
        nxt = max(int(k) for k in self.state) + 1
        print(f'-> After a disconnect, re-run cells 1..{nxt-1} (they are cached '
              f'and fast), then continue at cell {nxt}.')

---
## Cell 1 — Setup

In [ ]:
# ============================ CELL 1 — SETUP ============================
if 'ResumableCollector' not in globals(): raise SystemExit('>>> Run CELL 0 first.')
!pip -q install linearmodels rdrobust pandas numpy scipy matplotlib requests pyarrow tqdm statsmodels 2>/dev/null

import os, json, time, re, subprocess, urllib.parse, urllib.request
import datetime as dt
from pathlib import Path
import numpy as np, pandas as pd, requests
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

try:
    from google.colab import drive; drive.mount('/content/drive')
    WORK = Path('/content/drive/MyDrive/msr2027')
except Exception:
    WORK = Path('./msr2027')
CACHE = WORK/'cache'; CACHE.mkdir(parents=True, exist_ok=True)

ANNOUNCE_DATE = dt.date(2022,7,8); T0 = pd.Timestamp(ANNOUNCE_DATE)
COHORT_SIZE   = 3800
CONTACT_EMAIL = 'anonymous.artifact@example.com'
PANEL_START, PANEL_END = '2021-01', '2023-12'   # ends before universal 2FA (Jan 2024)

STATE = CellState(CACHE/'checkpoint_stage3.json'); STATE.report()

full = pd.read_parquet(CACHE/'analysis_frame_rdd.parquet')
d2   = json.loads((CACHE/'results_rdd.json').read_text())
print(f"RDD frame: {len(full):,} projects "
      f"({int(full.treated.sum()):,} treated / {int((1-full.treated).sum()):,} control)")
print(f"RDD:   est={rddres['primary_estimate']:+.4f}  p={rddres['primary_robust_p']:.3f}  "
      f"MDE={rddres['mde_80pct']:.3f} ({rddres['mde_80pct']/rddres['control_base_rate']:.0%} of base)")

CH='https://sql-clickhouse.clickhouse.com'
def ch(sql, fmt='JSON', timeout=600):
    p=urllib.parse.urlencode({'user':'demo','default_format':fmt})
    req=urllib.request.Request(CH+'?'+p, data=sql.encode(), method='POST')
    return urllib.request.urlopen(req, timeout=timeout).read().decode()

STATE.mark(1); SETUP_DONE=True; print('\nCELL 1 OK')

---
## Cell 2 — Identify late entrants (to exclude them)

For each month from Aug 2022 to Dec 2023, reconstruct the trailing-window top-3,800. Any control-side project that appears in *any* of those rankings might have been designated later — its treatment status is ambiguous, so it leaves the sample entirely. Cached per month; safe to re-run.

In [ ]:
# =============== CELL 2 — LATE ENTRANTS (EXCLUSIONS) ====================
if not globals().get('SETUP_DONE'): raise SystemExit('>>> Run CELL 1 first.')

LATE = CACHE/'late_entrants.parquet'
if LATE.exists():
    late = set(pd.read_parquet(LATE)['project'])
    print(f'Loaded cached late-entrant set: {len(late):,}')
else:
    months_asof = pd.date_range('2022-08-01','2023-12-01',freq='MS')
    initial = set(full.loc[full.treated==1,'project'])
    late = set()
    for m in tqdm(months_asof):
        f = CACHE/f'top_{m:%Y_%m}.json'
        if f.exists():
            top = set(json.loads(f.read_text()))
        else:
            win = [(m - pd.DateOffset(months=k)).strftime('%Y-%m-01') for k in range(0,7)]
            sql = f"""SELECT project, SUM(count) AS n FROM pypi.pypi_downloads_per_month
                      WHERE month IN ({','.join(repr(w) for w in win)})
                      GROUP BY project ORDER BY n DESC LIMIT {COHORT_SIZE}"""
            top = {r['project'] for r in json.loads(ch(sql))['data']}
            f.write_text(json.dumps(sorted(top)))
        late |= (top - initial)
    late &= set(full['project'])            # only those in our frame matter
    pd.DataFrame({'project':sorted(late)}).to_parquet(LATE)
    print(f'Late entrants found in our frame: {len(late):,}')

frame = full[~full['project'].isin(late)].copy()
print(f'Analysis sample: {len(frame):,} projects '
      f'({int(frame.treated.sum()):,} treated July-2022 cohort / '
      f'{int((1-frame.treated).sum()):,} never-top-3800 controls)')
print(f'Excluded as ambiguous: {len(full)-len(frame):,}')
STATE.mark(2); EXCL_DONE=True; print('\nCELL 2 OK')

---
## Cell 3 — Collect the monthly release panel (resumable)

Stage 2 stored summary statistics, not timestamps, so the month-by-month panel needs one more pass over PyPI. Same two-API logic, same crash-safe collector — interrupt any time, re-run to resume. This records, per project, the set of months with at least one release.

In [ ]:
# ================ CELL 3 — MONTHLY PANEL COLLECTION =====================
if not globals().get('EXCL_DONE'): raise SystemExit('>>> Run CELL 2 first.')

sess = requests.Session()
sess.headers.update({'User-Agent': f'MSR2027-research ({CONTACT_EMAIL})'})

def upload_times(pr):
    try:
        r=sess.get(f'https://pypi.org/pypi/{pr}/json', timeout=25)
        if r.status_code==200:
            rel=r.json().get('releases')
            if rel:
                ts=[pd.Timestamp(f['upload_time_iso_8601']).tz_localize(None)
                    for v in rel.values() for f in v[:1] if f.get('upload_time_iso_8601')]
                if ts: return sorted(ts)
    except Exception: pass
    try:
        r=sess.get(f'https://pypi.org/simple/{pr}/', timeout=25,
                   headers={'Accept':'application/vnd.pypi.simple.v1+json'})
        if r.status_code==200:
            ts=[pd.Timestamp(f['upload-time']).tz_localize(None)
                for f in r.json().get('files',[]) if f.get('upload-time')]
            if ts: return sorted(ts)
    except Exception: pass
    return None

def rec_months(pr):
    ts = upload_times(pr)
    if not ts: return None
    return {'months': sorted({t.strftime('%Y-%m') for t in ts})}

COL = ResumableCollector('pypi_month_panel',
                         local_dir='/content/msr_state',
                         mirror_dir=str(CACHE/'state'),
                         workers=12, flush_every=10,
                         mirror_every=100, mirror_seconds=60)
gotm = COL.run(list(frame['project']), rec_months)
print(f'\nMonthly data for {len(gotm):,} / {len(frame):,}')
STATE.mark(3); PANEL_RAW_DONE=True; print('\nCELL 3 OK')

---
## Cell 4 — Build the panel

One row per project per month, Jan 2021 – Dec 2023. Outcome: did the project release anything that month. Also prints the raw treated-vs-control monthly series — **look at it**. Parallel pre-trends should be visible to the eye before any regression asserts them.

In [ ]:
# ===================== CELL 4 — PANEL CONSTRUCTION ======================
if not globals().get('PANEL_RAW_DONE'): raise SystemExit('>>> Run CELL 3 first.')

months = pd.period_range(PANEL_START, PANEL_END, freq='M').astype(str)
mset = {p:set(m) for p,m in zip(gotm['project'], gotm['months'])}
tr   = dict(zip(frame['project'], frame['treated']))

rows=[]
for p in frame['project']:
    if p not in mset: continue
    s=mset[p]; t=tr[p]
    for mm in months:
        rows.append((p, mm, int(mm in s), int(t)))
panel = pd.DataFrame(rows, columns=['project','month','released','treated'])
panel['post'] = (panel['month'] >= '2022-07').astype(int)

n_proj = panel['project'].nunique()
print(f'Panel: {len(panel):,} project-months, {n_proj:,} projects, {len(months)} months')

g = panel.groupby(['month','treated'])['released'].mean().unstack()
fig,ax=plt.subplots(figsize=(11,4))
ax.plot(g.index, g[0], label='control (never top-3800)', color='#4C78A8')
ax.plot(g.index, g[1], label='treated (July 2022 cohort)', color='#F58518')
ax.axvline(list(g.index).index('2022-07'), color='crimson', ls='--', lw=1.5, label='designation')
ax.set_xticks(range(0,len(g.index),3)); ax.set_xticklabels(g.index[::3], rotation=45)
ax.set_ylabel('share releasing this month'); ax.legend(); ax.set_title('Raw monthly release rates')
plt.tight_layout(); plt.savefig(CACHE/'fig_raw_trends.png', dpi=160); plt.show()

pre = g[g.index < '2022-07']
print('\nPre-period means  treated={:.3f}  control={:.3f}  gap={:+.3f}'.format(
      pre[1].mean(), pre[0].mean(), pre[1].mean()-pre[0].mean()))
print('Pre-period trend corr(treated, control) = {:.3f}'.format(pre[1].corr(pre[0])))
STATE.mark(4); PANEL_DONE=True; print('\nCELL 4 OK')

---
## Cell 5 — Event study and DiD estimate

Two-way fixed effects — project and month — with standard errors clustered by project. Because there is a **single treatment cohort with never-treated controls**, none of the staggered-adoption pathologies apply; TWFE is the correct estimator here, not a compromise.

The event study is the honest version of the result: one coefficient per month relative to June 2022. The leads (pre-treatment) are the credibility test — they should sit on zero. The lags are the finding.

In [ ]:
# ================= CELL 5 — EVENT STUDY + DiD ===========================
if not globals().get('PANEL_DONE'): raise SystemExit('>>> Run CELL 4 first.')
from linearmodels.panel import PanelOLS

pan = panel.copy()
pan['mnum'] = pan['month'].map({m:i for i,m in enumerate(months)})
t0i = list(months).index('2022-07')
pan['k'] = pan['mnum'] - t0i                       # event time; treatment at k=0

pan = pan.set_index(['project','mnum'])

# ---- (a) simple DiD ----------------------------------------------------
pan['D'] = (pan['treated']*pan['post']).astype(float)
m1 = PanelOLS(pan['released'], pan[['D']], entity_effects=True, time_effects=True
     ).fit(cov_type='clustered', cluster_entity=True)
ATT  = float(m1.params['D']); SE = float(m1.std_errors['D']); PV = float(m1.pvalues['D'])
print('='*70); print('DiD (TWFE, single cohort, never-treated controls)'); print('='*70)
print(f'  ATT = {ATT:+.5f}   cluster-robust SE = {SE:.5f}   p = {PV:.4f}')

# ---- (b) event study ---------------------------------------------------
K_MIN, K_MAX = -12, 12
ev = pan[(pan['k']>=K_MIN)&(pan['k']<=K_MAX)].copy()
X = {}
for k in range(K_MIN, K_MAX+1):
    if k == -1: continue                          # omitted category
    X[f'k{k}'] = ((ev['k']==k)&(ev['treated']==1)).astype(float)
X = pd.DataFrame(X, index=ev.index)
m2 = PanelOLS(ev['released'], X, entity_effects=True, time_effects=True
     ).fit(cov_type='clustered', cluster_entity=True)

ks   = [k for k in range(K_MIN,K_MAX+1) if k!=-1]
coef = [float(m2.params[f'k{k}']) for k in ks]
ci_l = [float(m2.conf_int().loc[f'k{k}','lower']) for k in ks]
ci_u = [float(m2.conf_int().loc[f'k{k}','upper']) for k in ks]

fig,ax=plt.subplots(figsize=(11,4))
ax.errorbar(ks, coef, yerr=[np.array(coef)-np.array(ci_l), np.array(ci_u)-np.array(coef)],
            fmt='o', color='#4C78A8', capsize=3)
ax.axhline(0,color='gray',lw=1); ax.axvline(-0.5,color='crimson',ls='--',lw=1.5)
ax.set_xlabel('months since designation'); ax.set_ylabel('effect on P(release)')
ax.set_title('Event study — leads should sit on zero')
plt.tight_layout(); plt.savefig(CACHE/'fig_event_study.png', dpi=160); plt.show()

leads = [k for k in ks if k < -1]
lead_sig = sum(1 for k in leads
               if float(m2.pvalues[f'k{k}']) < 0.05)
print(f'\nPre-trend check: {lead_sig}/{len(leads)} lead coefficients significant at 5% '
      f'(expect ~{0.05*len(leads):.1f} by chance)')
mean_post = float(np.mean([float(m2.params[f'k{k}']) for k in ks if k>=0]))
print(f'Mean post coefficient (k=0..{K_MAX}): {mean_post:+.5f}')
STATE.mark(5); EST_DONE=True; print('\nCELL 5 OK')

---
## Cell 6 — Power and adequacy

A null result is informative only when the design could have detected a
real effect. This cell reports the panel design's minimum detectable
effect and the pre-trend diagnostics against two pre-committed adequacy
criteria: an MDE at or below 15% of the base rate, and clean
pre-treatment trends. Both must hold for the panel estimate to serve as
the primary result.

In [ ]:
# ================= CELL 6 — POWER + ADEQUACY ==========================
if not globals().get('EST_DONE'): raise SystemExit('>>> Run CELL 5 first.')
from scipy import stats

base_m = panel.loc[(panel.treated==0)&(panel.post==0),'released'].mean()
MDE = (stats.norm.ppf(0.975)+stats.norm.ppf(0.80))*SE
rel = MDE/base_m

print(f'  control pre-period monthly release rate : {base_m:.4f}')
print(f'  DiD cluster-robust SE                   : {SE:.5f}')
print(f'  MDE (80% power, a=0.05)                 : {MDE:.4f}  = {rel:.1%} of base')
print(f'  vs RDD MDE                              : '
      f'{rddres["mde_80pct"]/rddres["control_base_rate"]:.0%} of base')
print(f'  observed ATT                            : {ATT:+.5f} (p={PV:.4f}, {ATT/base_m:+.1%} of base)')

pre_ok  = lead_sig <= max(1, round(0.10*len(leads)))
powered = rel <= 0.15
print('\n'+'='*70); print('  ADEQUACY CHECK'); print('='*70)
print(f'  [{"PASS" if powered else "FAIL"}]  MDE <= 15% of base rate')
print(f'  [{"PASS" if pre_ok  else "FAIL"}]  pre-trends clean')
if powered and pre_ok:
    print('\n  >>> Design adequate: the panel estimate is the primary result.')
    if PV>=0.05:
        print(f'      Informative null: effects larger than {rel:.0%} of the base rate are excluded.')
    else:
        print(f'      Significant ATT of {ATT/base_m:+.1%}; verify it is visible in the raw')
        print('      series of Cell 4 before interpreting.')
else:
    print('\n  >>> Design inadequate for an informative estimate on this dataset.')
GATE_DONE=True
STATE.mark(6); print('\nCELL 6 OK')

---
## Cell 7 — Export

In [ ]:
# ======================= CELL 7 — EXPORT ================================
if not globals().get('GATE_DONE'): raise SystemExit('>>> Run CELL 6 first.')
out = {
 'run_at': dt.datetime.now(dt.timezone.utc).isoformat(),
 'n_projects': int(panel['project'].nunique()),
 'n_project_months': int(len(panel)),
 'n_treated': int(frame.treated.sum()),
 'n_control': int((1-frame.treated).sum()),
 'n_late_excluded': int(len(full)-len(frame)),
 'att': ATT, 'att_se': SE, 'att_p': PV,
 'base_monthly_rate': float(base_m),
 'mde_80pct': float(MDE), 'mde_rel_base': float(rel),
 'lead_sig': int(lead_sig), 'n_leads': int(len(leads)),
 'mean_post_coef': float(mean_post),
 'rdd_stage2': d2,
}
(CACHE/'results_panel.json').write_text(json.dumps(out, indent=2))
print(json.dumps(out, indent=2))
print('\nSaved results_panel.json (+ fig_raw_trends.png, fig_event_study.png).')